# 作业四

**学号：** 20234080404  
**姓名：** 刘留

## 2 语言模型

### 2.1 理论计算题

给定一个字符序列 **"ababc"**，假设我们使用一阶马尔可夫语言模型（即 $p(x_t \mid x_{t-1})$），
并使用加 1 平滑（Add-1 Smoothing）。词汇表为 $\{'a', 'b', 'c'\}$。

**请计算：**
1. $p(\text{'a'} \mid \text{'b'})$
2. $p(\text{'c'} \mid \text{'b'})$

计算时考虑所有可能的转移情况，包括未见过的转移对。

**解：**

**步骤 1：统计序列中的转移计数**

序列 "ababc" 中相邻字符对：
- a → b（位置 0→1）：1 次
- b → a（位置 1→2）：1 次
- a → b（位置 2→3）：1 次
- b → c（位置 3→4）：1 次

统计条件为 "b" 的转移：
- $\text{count}(b \to a) = 1$
- $\text{count}(b \to b) = 0$
- $\text{count}(b \to c) = 1$
- $\text{count}(b) = \text{count}(b \to a) + \text{count}(b \to b) + \text{count}(b \to c) = 1 + 0 + 1 = 2$

**步骤 2：加 1 平滑（Add-1 Smoothing）**

词汇表大小 $V = 3$（a, b, c）。

加 1 平滑公式：
$$p_{\text{smooth}}(w \mid b) = \frac{\text{count}(b \to w) + 1}{\text{count}(b) + V}$$

**1. $p(\text{'a'} \mid \text{'b'})$：**
$$p(a \mid b) = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4$$

**2. $p(\text{'c'} \mid \text{'b'})$：**
$$p(c \mid b) = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4$$

（补充：$p(b \mid b) = \frac{0 + 1}{2 + 3} = \frac{1}{5} = 0.2$）

### 2.2 编程题

请编写一个函数 `preprocess_text(text, n)`，完成以下步骤：

1. 将文本转换为小写，移除标点符号（保留空格和字母）；
2. 按空格进行分词；
3. 构建词汇表（按词出现频率排序，分配整数 ID，从 0 开始）；
4. 使用大小为 n 的滑动窗口生成 n-gram 特征序列和对应的下一个词目标（用于自回归语言模型）。

返回词汇表和 (特征表, 目标表)。例如，输入 "The time machine" 和 n=2，应生成特征
`[['the', 'time'], ['time', 'machine']]` 和目标 `['machine', None]`（末尾位置无下一个词，用 None 填充）。

In [1]:
import re
from collections import Counter


def preprocess_text(text, n):
    """
    文本预处理函数：构建词汇表并生成 n-gram 特征和目标。

    参数：
        text: 输入文本字符串
        n: n-gram 窗口大小

    返回：
        vocab: 词汇表字典 {词: ID}
        features: n-gram 特征列表，每个元素为 n 个词的列表
        targets: 目标词列表，对应每个特征的下一个词（末尾为 None）
    """
    # 1. 转小写并移除标点（保留字母、数字和空格）
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

    # 2. 按空格分词
    tokens = text.split()
    if not tokens:
        return {}, [], []

    # 3. 构建词汇表（按频率降序排列，从 0 开始分配 ID）
    token_counts = Counter(tokens)
    sorted_tokens = sorted(token_counts.items(), key=lambda x: (-x[1], x[0]))
    vocab = {token: idx for idx, (token, _) in enumerate(sorted_tokens)}

    # 4. 滑动窗口生成 n-gram 特征和目标
    features = []
    targets = []
    for i in range(len(tokens) - n + 1):
        # n-gram features
        feature = tokens[i:i+n]
        features.append(feature)
        # target is the next word, or None if at the end
        if i + n < len(tokens):
            targets.append(tokens[i + n])
        else:
            targets.append(None)

    return vocab, features, targets


# 测试
if __name__ == '__main__':
    text = "The time machine"
    n = 2
    vocab, features, targets = preprocess_text(text, n)
    print(f'词汇表: {vocab}')
    print(f'特征: {features}')
    print(f'目标: {targets}')
    print()

    # 更复杂的测试
    text2 = "The quick brown fox jumps over the lazy dog. The dog is very lazy!"
    n2 = 3
    vocab2, features2, targets2 = preprocess_text(text2, n2)
    print(f'词汇表: {vocab2}')
    print(f'特征数量: {len(features2)}')
    print(f'前3个特征: {features2[:3]}')
    print(f'前3个目标: {targets2[:3]}')

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征: [['the', 'time'], ['time', 'machine']]
目标: ['machine', None]

词汇表: {'the': 0, 'dog': 1, 'lazy': 2, 'brown': 3, 'fox': 4, 'is': 5, 'jumps': 6, 'over': 7, 'quick': 8, 'very': 9}
特征数量: 12
前3个特征: [['the', 'quick', 'brown'], ['quick', 'brown', 'fox'], ['brown', 'fox', 'jumps']]
前3个目标: ['fox', 'jumps', 'over']


## 3 循环神经网络

### 3.1 理论计算题

考虑一个线性 RNN（无偏置），其定义为：
$$h_t = W_{hh} h_{t-1} + W_{hx} x_t$$
$$o_t = W_{oh} h_t$$

假设损失函数为均方误差：
$$L = \frac{1}{2} \sum_{t=1}^{T} (o_t - y_t)^2$$

**请写出损失对权重 $W_{hh}$ 的梯度表达式（通过时间反向传播，展开到所有时间步），并说明梯度消失或爆炸的条件。**

**解：**

**（1）梯度推导**

记 $e_t = o_t - y_t$ 为第 $t$ 步的输出误差。

损失函数对 $o_t$ 的偏导：
$$\frac{\partial L}{\partial o_t} = o_t - y_t = e_t$$

首先计算 $\frac{\partial L}{\partial h_t}$。由于 $h_t$ 通过两条路径影响损失：
- 直接影响 $o_t$
- 通过 $h_{t+1}$ 间接影响后续输出

定义 $\delta_t = \frac{\partial L}{\partial h_t}$，则：
$$\delta_T = \frac{\partial L}{\partial o_T} \cdot \frac{\partial o_T}{\partial h_T} = (o_T - y_T) \cdot W_{oh}^T$$

对于 $t < T$（反向递归）：
$$\delta_t = \frac{\partial L}{\partial o_t} \frac{\partial o_t}{\partial h_t} + \frac{\partial L}{\partial h_{t+1}} \frac{\partial h_{t+1}}{\partial h_t} = W_{oh}^T e_t + W_{hh}^T \delta_{t+1}$$

展开得到：
$$\delta_t = W_{oh}^T e_t + W_{hh}^T W_{oh}^T e_{t+1} + (W_{hh}^T)^2 W_{oh}^T e_{t+2} + \cdots + (W_{hh}^T)^{T-t} W_{oh}^T e_T$$

即：
$$\delta_t = \sum_{k=t}^{T} (W_{hh}^T)^{k-t} W_{oh}^T e_k$$

最后，$L$ 对 $W_{hh}$ 的梯度：
$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \frac{\partial L}{\partial h_t} \frac{\partial h_t}{\partial W_{hh}} = \sum_{t=1}^{T} \delta_t h_{t-1}^T$$

将 $\delta_t$ 代入：
$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \left[ \sum_{k=t}^{T} (W_{hh}^T)^{k-t} W_{oh}^T e_k \right] h_{t-1}^T$$

**（2）梯度消失/爆炸的条件**

梯度中反复出现 $(W_{hh}^T)^{k-t}$ 因子。设 $W_{hh}$ 的特征值分解为 $W_{hh} = Q \Lambda Q^{-1}$，其中 $\Lambda = \text{diag}(\lambda_1, \ldots, \lambda_h)$。

则 $(W_{hh}^T)^{k-t} = (Q^{-1})^T \Lambda^{k-t} Q^T$。

- **梯度爆炸条件：** 若 $W_{hh}$ 的最大特征值 $|\lambda_{\max}| > 1$，则 $\Lambda^{k-t}$ 随 $k-t$ 指数增长，导致梯度爆炸。
- **梯度消失条件：** 若 $W_{hh}$ 的最大特征值 $|\lambda_{\max}| < 1$，则 $\Lambda^{k-t}$ 随 $k-t$ 指数衰减至 0，导致梯度消失。
- **稳定条件：** 只有当 $|\lambda_{\max}| = 1$ 时，梯度才能在长序列中稳定传播。

### 3.2 编程题

实现一个简单的 RNN 单元的前向传播和单步反向传播（计算梯度，不更新）。给定输入 $x_t$（形状 `(batch_size, input_size)`）、上一隐藏状态 $h_{prev}$（形状 `(batch_size, hidden_size)`），以及权重 $W_{hx}$、$W_{hh}$、$b_h$，计算当前隐藏状态 $h_t$。同时实现反向传播，已知上游梯度 $dh_{next}$（即损失对 $h_t$ 的梯度），计算 $dx_t$、$dh_{prev}$、$dW_{hx}$、$dW_{hh}$、$db_h$。使用 tanh 激活函数。

In [2]:
import torch


def rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN 单元前向传播。

    参数：
        x_t: 输入，形状 (batch_size, input_size)
        h_prev: 上一隐藏状态，形状 (batch_size, hidden_size)
        W_hx: 输入到隐藏权重，形状 (input_size, hidden_size)
        W_hh: 隐藏到隐藏权重，形状 (hidden_size, hidden_size)
        b_h: 隐藏层偏置，形状 (hidden_size,)

    返回：
        h_t: 当前隐藏状态，形状 (batch_size, hidden_size)
        cache: 中间结果缓存，用于反向传播
    """
    # 线性组合
    a_t = x_t @ W_hx + h_prev @ W_hh + b_h  # (batch_size, hidden_size)
    # tanh 激活
    h_t = torch.tanh(a_t)

    cache = (x_t, h_prev, a_t, h_t)
    return h_t, cache


def rnn_cell_backward(dh_next, cache):
    """
    RNN 单元单步反向传播（不包含参数更新，仅计算梯度）。

    参数：
        dh_next: 损失对 h_t 的梯度，形状 (batch_size, hidden_size)
        cache: 来自前向传播的中间结果

    返回：
        dx_t: 损失对 x_t 的梯度，形状 (batch_size, input_size)
        dh_prev: 损失对 h_prev 的梯度，形状 (batch_size, hidden_size)
        dW_hx: 损失对 W_hx 的梯度，形状 (input_size, hidden_size)
        dW_hh: 损失对 W_hh 的梯度，形状 (hidden_size, hidden_size)
        db_h: 损失对 b_h 的梯度，形状 (hidden_size,)
    """
    x_t, h_prev, a_t, h_t = cache

    # tanh 反向：d(tanh(z))/dz = 1 - tanh(z)^2
    da_t = dh_next * (1 - h_t ** 2)  # (batch_size, hidden_size)

    # 计算各梯度
    dx_t = da_t @ W_hx.T              # (batch_size, input_size)
    dh_prev = da_t @ W_hh.T           # (batch_size, hidden_size)
    dW_hx = x_t.T @ da_t              # (input_size, hidden_size)
    dW_hh = h_prev.T @ da_t           # (hidden_size, hidden_size)
    db_h = da_t.sum(dim=0)            # (hidden_size,)

    return dx_t, dh_prev, dW_hx, dW_hh, db_h


# ====== 测试 ======
torch.manual_seed(42)

batch_size = 4
input_size = 10
hidden_size = 8

# 随机输入和参数
x_t = torch.randn(batch_size, input_size)
h_prev = torch.randn(batch_size, hidden_size)
W_hx = torch.randn(input_size, hidden_size)
W_hh = torch.randn(hidden_size, hidden_size)
b_h = torch.randn(hidden_size)

# 前向传播
h_t, cache = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h)
print(f'输入 x_t 形状: {x_t.shape}')
print(f'上一隐藏状态 h_prev 形状: {h_prev.shape}')
print(f'输出 h_t 形状: {h_t.shape}')

# 模拟上游梯度
dh_next = torch.randn(batch_size, hidden_size)
dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_cell_backward(dh_next, cache)

print(f'\ndx_t 形状: {dx_t.shape}')
print(f'dh_prev 形状: {dh_prev.shape}')
print(f'dW_hx 形状: {dW_hx.shape}')
print(f'dW_hh 形状: {dW_hh.shape}')
print(f'db_h 形状: {db_h.shape}')

# ====== 数值梯度验证 ======
print('\n===== 数值梯度验证 =====')

# 使用 PyTorch autograd 验证
x_t_grad = x_t.detach().clone().requires_grad_(True)
h_prev_grad = h_prev.detach().clone().requires_grad_(True)
W_hx_grad = W_hx.detach().clone().requires_grad_(True)
W_hh_grad = W_hh.detach().clone().requires_grad_(True)
b_h_grad = b_h.detach().clone().requires_grad_(True)

a_t_grad = x_t_grad @ W_hx_grad + h_prev_grad @ W_hh_grad + b_h_grad
h_t_grad = torch.tanh(a_t_grad)
# 用与 dh_next 相同的方式计算一个标量损失
loss = (h_t_grad * dh_next).sum()
loss.backward()

print(f'dW_hx 一致: {torch.allclose(dW_hx, W_hx_grad.grad, atol=1e-6)}')
print(f'dW_hh 一致: {torch.allclose(dW_hh, W_hh_grad.grad, atol=1e-6)}')
print(f'db_h 一致: {torch.allclose(db_h, b_h_grad.grad, atol=1e-6)}')
print(f'dx_t 一致: {torch.allclose(dx_t, x_t_grad.grad, atol=1e-6)}')
print(f'dh_prev 一致: {torch.allclose(dh_prev, h_prev_grad.grad, atol=1e-6)}')

输入 x_t 形状: torch.Size([4, 10])
上一隐藏状态 h_prev 形状: torch.Size([4, 8])
输出 h_t 形状: torch.Size([4, 8])

dx_t 形状: torch.Size([4, 10])
dh_prev 形状: torch.Size([4, 8])
dW_hx 形状: torch.Size([10, 8])
dW_hh 形状: torch.Size([8, 8])
db_h 形状: torch.Size([8])

===== 数值梯度验证 =====
dW_hx 一致: True
dW_hh 一致: True
db_h 一致: True
dx_t 一致: True
dh_prev 一致: True


## 4 深层循环神经网络

### 4.1 理论计算题

假设一个深度双向 RNN，有 $L$ 层，每层隐藏单元数为 $H$，输入维度为 $D$，输出维度为 $O$（仅考虑最后的输出层）。计算该模型的参数总数（包括所有全连接层的权重和偏置），忽略输入层和输出层之间的影响，明确给出表达式。

**解：**

**逐层分析参数：**

**第 1 层（双向 RNN，接收输入 $D$）：**
- 前向 RNN：输入权重 $W_{hx}^{(1,f)} \in \mathbb{R}^{D \times H}$，隐藏权重 $W_{hh}^{(1,f)} \in \mathbb{R}^{H \times H}$
  偏置 $b_h^{(1,f)} \in \mathbb{R}^{H}$（每个方向各一个偏置）
- 后向 RNN：输入权重 $W_{hx}^{(1,b)} \in \mathbb{R}^{D \times H}$，隐藏权重 $W_{hh}^{(1,b)} \in \mathbb{R}^{H \times H}$
  偏置 $b_h^{(1,b)} \in \mathbb{R}^{H}$

参数数量：$2 \times (DH + H^2 + H) = 2DH + 2H^2 + 2H$

**第 $l$ 层（$l = 2, \ldots, L$，双向 RNN，从上一层接收 $2H$ 维输入（前向+后向拼接））：**
- 输入维度为 $2H$（上一层的双向输出拼接）
- 前向 RNN：输入权重 $\in \mathbb{R}^{2H \times H}$，隐藏权重 $\in \mathbb{R}^{H \times H}$，偏置 $\in \mathbb{R}^{H}$
- 后向 RNN：输入权重 $\in \mathbb{R}^{2H \times H}$，隐藏权重 $\in \mathbb{R}^{H \times H}$，偏置 $\in \mathbb{R}^{H}$

参数数量：$2 \times (2H \cdot H + H^2 + H) = 2 \times (3H^2 + H) = 6H^2 + 2H$

**输出层（从最后一层双向输出 $2H$ 映射到 $O$）：**
- 权重 $W_{out} \in \mathbb{R}^{2H \times O}$，偏置 $b_{out} \in \mathbb{R}^{O}$

参数数量：$2HO + O$

**总参数量：**
$$\begin{aligned}
\text{Params}_{\text{total}} &= [2DH + 2H^2 + 2H] + (L-1) \times [6H^2 + 2H] + [2HO + O] \\
&= 2DH + 2H^2 + 2H + 6(L-1)H^2 + 2(L-1)H + 2HO + O \\
&= 2DH + (6L - 4)H^2 + 2LH + 2HO + O
\end{aligned}$$

**简化形式（按权重和偏置分开）：**
- 权重参数：$2DH + 2H^2 + (L-1) \cdot 6H^2 + 2HO = 2DH + (6L - 4)H^2 + 2HO$
- 偏置参数：$2H + 2(L-1)H + O = 2LH + O$

### 4.2 编程题

实现一个双向 RNN 编码器，接收序列 $X$（形状 `(seq_len, batch, input_dim)`），使用 `torch.nn.RNN` 或手动实现。要求返回每个时间步的拼接（前向+后向）隐藏状态（形状 `(seq_len, batch, 2*hidden_dim)`），以及最终时间步的拼接隐藏状态（作为序列表示）。

In [3]:
import torch
import torch.nn as nn


class BiRNNEncoder(nn.Module):
    """
    双向 RNN 编码器。

    参数：
        input_dim: 输入特征维度
        hidden_dim: 隐藏层维度（每个方向）
        num_layers: RNN 层数
    """
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # 使用 PyTorch 内置双向 RNN
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=False  # 输入形状 (seq_len, batch, input_dim)
        )

    def forward(self, X):
        """
        参数：
            X: 输入序列，形状 (seq_len, batch, input_dim)

        返回：
            all_hidden: 每个时间步的拼接隐藏状态，
                        形状 (seq_len, batch, 2 * hidden_dim)
            final_hidden: 最终时间步的拼接隐藏状态（序列表示），
                          形状 (batch, 2 * hidden_dim)
        """
        # RNN 输出：
        #   output: (seq_len, batch, 2 * hidden_dim) — 每个时间步的输出
        #   h_n: (2 * num_layers, batch, hidden_dim) — 最终隐藏状态
        output, h_n = self.rnn(X)

        # 最终时间步的隐藏状态（取最后一层的双向隐藏状态拼接）
        # h_n 形状: (2 * num_layers, batch, hidden_dim)
        # 最后两层为最后一层的前向和后向
        h_forward_final = h_n[-2]  # 最后一层前向，形状 (batch, hidden_dim)
        h_backward_final = h_n[-1]  # 最后一层后向，形状 (batch, hidden_dim)
        final_hidden = torch.cat([h_forward_final, h_backward_final], dim=-1)

        return output, final_hidden


# ====== 测试 ======
torch.manual_seed(42)

seq_len, batch_size, input_dim = 6, 3, 10
hidden_dim = 8

X = torch.randn(seq_len, batch_size, input_dim)

# 创建编码器
encoder = BiRNNEncoder(input_dim=input_dim, hidden_dim=hidden_dim, num_layers=1)
all_hidden, final_hidden = encoder(X)

print(f'输入 X 形状: {X.shape}')
print(f'每个时间步输出形状: {all_hidden.shape}')
print(f'最终隐藏状态形状: {final_hidden.shape}')
print()

# 验证输出维度
assert all_hidden.shape == (seq_len, batch_size, 2 * hidden_dim),     f'期望形状 ({seq_len}, {batch_size}, {2 * hidden_dim})，实际 {all_hidden.shape}'
assert final_hidden.shape == (batch_size, 2 * hidden_dim),     f'期望形状 ({batch_size}, {2 * hidden_dim})，实际 {final_hidden.shape}'
print('维度验证通过！')

# 检查输出值（确保双向拼接正确）
print(f'\nall_hidden 前 3 个元素: {all_hidden[:3, 0, 0]}')
print(f'final_hidden 前 3 个元素: {final_hidden[0, :3]}')

# 统计参数量
total_params = sum(p.numel() for p in encoder.parameters())
print(f'\n总参数量: {total_params}')

输入 X 形状: torch.Size([6, 3, 10])
每个时间步输出形状: torch.Size([6, 3, 16])
最终隐藏状态形状: torch.Size([3, 16])

维度验证通过！

all_hidden 前 3 个元素: tensor([-0.7820, -0.0917, -0.1708], grad_fn=<SelectBackward0>)
final_hidden 前 3 个元素: tensor([-0.7705,  0.9135,  0.8594], grad_fn=<SliceBackward0>)

总参数量: 320


## 5 嵌入向量

### 5.1 理论计算题

在 Skip-gram 模型中，给定中心词 $w_c$ 和上下文词 $w_o$，使用负采样（采样 $K$ 个负样本）。请写出其损失函数（对单个样本对）的表达式，并说明如何从噪声分布中采样负样本。假设中心词向量为 $\mathbf{v}_c$，上下文词向量为 $\mathbf{u}_o$，负样本词向量为 $\mathbf{u}_{n_k}$，写出完整的目标函数。

**解：**

**（1）损失函数**

Skip-gram 负采样对一对中心词-上下文词 $(w_c, w_o)$ 的目标是最大化正样本概率并最小化 $K$ 个负样本概率：

$$\mathcal{L} = -\log \sigma(\mathbf{u}_o^T \mathbf{v}_c) - \sum_{k=1}^{K} \mathbb{E}_{n_k \sim P_n(w)} \left[ \log \sigma(-\mathbf{u}_{n_k}^T \mathbf{v}_c) \right]$$

其中 $\sigma(x) = \frac{1}{1 + e^{-x}}$ 为 sigmoid 函数。

对于具体的 $K$ 个负样本 $n_1, n_2, \ldots, n_K$，损失函数为：

$$\mathcal{L} = -\log \sigma(\mathbf{u}_o^T \mathbf{v}_c) - \sum_{k=1}^{K} \log \sigma(-\mathbf{u}_{n_k}^T \mathbf{v}_c)$$

也可以等价地写为（最小化负对数似然）：

$$\mathcal{L} = -\left[ \log \sigma(\mathbf{u}_o^T \mathbf{v}_c) + \sum_{k=1}^{K} \log(1 - \sigma(\mathbf{u}_{n_k}^T \mathbf{v}_c)) \right]$$

**（2）负采样策略**

负样本从噪声分布 $P_n(w)$ 中采样。常用的噪声分布为词频的 $3/4$ 次幂（即 Unigram 分布的平滑版本）：

$$P_n(w) = \frac{\text{count}(w)^{3/4}}{\sum_{w'} \text{count}(w')^{3/4}}$$

采样过程：
1. 统计语料中每个词的频数 $\text{count}(w)$；
2. 计算每个词的权重 $\text{count}(w)^{3/4}$；
3. 归一化得到概率分布 $P_n(w)$；
4. 按照 $P_n(w)$ 独立地采样 $K$ 个负样本词（通常排除正样本上下文词 $w_o$ 本身）。

使用 $3/4$ 次幂的目的是提高低频词被采样的概率，降低高频词被采样的概率，从而改善词嵌入的质量。

### 5.2 编程题

实现 CBOW 模型的前向传播和损失计算（不使用负采样，使用完整 softmax）。给定一批上下文词的索引表（每个样本有 `context_size` 个上下文词），词汇表大小 $V$，嵌入维度 $d$，输入权重矩阵 $W$（形状 `(V, d)`）和输出权重矩阵 $W_{out}$（形状 `(d, V)`）。计算平均上下文词向量作为隐藏层，然后计算输出概率分布，并计算交叉熵损失（目标为中心词索引）。返回损失值。

In [4]:
import torch
import torch.nn.functional as F


def cbow_forward_loss(context_indices, center_indices, W, W_out):
    """
    CBOW 模型前向传播和损失计算（完整 softmax）。

    参数：
        context_indices: 上下文词索引，形状 (batch_size, context_size)
                         每个样本包含 context_size 个上下文词索引
        center_indices: 中心词索引（目标），形状 (batch_size,)
        W: 输入嵌入矩阵（词 → 嵌入），形状 (V, d)
        W_out: 输出权重矩阵（嵌入 → 词），形状 (d, V)

    返回：
        loss: 标量交叉熵损失值
    """
    V, d = W.shape
    batch_size, context_size = context_indices.shape

    # 1. 查表获取上下文词嵌入，形状 (batch_size, context_size, d)
    context_embeddings = W[context_indices]

    # 2. 计算平均上下文词向量作为隐藏层，形状 (batch_size, d)
    h = context_embeddings.mean(dim=1)

    # 3. 计算 logits，形状 (batch_size, V)
    logits = h @ W_out  # (batch_size, d) @ (d, V) = (batch_size, V)

    # 4. 计算 log softmax
    log_probs = F.log_softmax(logits, dim=-1)

    # 5. 交叉熵损失（负对数似然）
    # 对每个样本取目标类别对应的 log 概率，取均值
    loss = -log_probs[torch.arange(batch_size), center_indices].mean()

    return loss


# ====== 测试 ======
torch.manual_seed(42)

V, d = 100, 50
batch_size = 8
context_size = 4

# 随机输入权重和输出权重
W = torch.randn(V, d)
W_out = torch.randn(d, V)

# 随机上下文词索引和中心词索引
context_indices = torch.randint(0, V, (batch_size, context_size))
center_indices = torch.randint(0, V, (batch_size,))

# 计算损失
loss = cbow_forward_loss(context_indices, center_indices, W, W_out)
print(f'词汇表大小 V = {V}, 嵌入维度 d = {d}')
print(f'批量大小 = {batch_size}, 上下文窗口 = {context_size}')
print(f'CBOW 损失: {loss.item():.6f}')

# ====== 验证 ======
print('\n===== 使用 PyTorch Embedding + Linear 验证 =====')

embedding = torch.nn.Embedding(V, d)
embedding.weight.data = W.clone()

linear = torch.nn.Linear(d, V, bias=False)
linear.weight.data = W_out.T.clone()  # Linear forward = x @ W^T

context_emb = embedding(context_indices)  # (batch_size, context_size, d)
h_verify = context_emb.mean(dim=1)        # (batch_size, d)
logits_verify = linear(h_verify)           # (batch_size, V)
loss_verify = F.cross_entropy(logits_verify, center_indices)

print(f'手写 CBOW 损失: {loss.item():.6f}')
print(f'PyTorch 验证损失: {loss_verify.item():.6f}')
print(f'结果一致: {torch.allclose(loss, loss_verify, atol=1e-6)}')

词汇表大小 V = 100, 嵌入维度 d = 50
批量大小 = 8, 上下文窗口 = 4
CBOW 损失: 8.751985

===== 使用 PyTorch Embedding + Linear 验证 =====
手写 CBOW 损失: 8.751985
PyTorch 验证损失: 8.751985
结果一致: True


## 6 注意力机制

### 6.1 理论计算题

给定查询矩阵 $Q \in \mathbb{R}^{2 \times 4}$，键矩阵 $K \in \mathbb{R}^{3 \times 4}$，值矩阵 $V \in \mathbb{R}^{3 \times 5}$。计算缩放点积注意力（单头）的输出矩阵，要求写出中间步骤（先计算得分矩阵，再 softmax，再加权求和）。使用 $\text{score} = QK^T / \sqrt{d_k}$，$d_k = 4$。

设具体数值为：

$$
Q = \begin{bmatrix}
1 & 0 & 1 & 0 \\
0 & 1 & 0 & 1
\end{bmatrix}, \quad
K = \begin{bmatrix}
1 & 0 & 1 & 0 \\
0 & 1 & 0 & 1 \\
1 & 1 & 0 & 0
\end{bmatrix}, \quad
V = \begin{bmatrix}
1 & 0 & 1 & 0 & 1 \\
0 & 1 & 0 & 1 & 0 \\
1 & 1 & 0 & 0 & 1
\end{bmatrix}
$$

可以写出数值计算过程，使用分数或具体数值。

**解：**

**步骤 1：计算得分矩阵 $S = QK^T / \sqrt{d_k}$**

$\sqrt{d_k} = \sqrt{4} = 2$

先计算 $QK^T$：

$$
QK^T = \begin{bmatrix}
1 & 0 & 1 & 0 \\
0 & 1 & 0 & 1
\end{bmatrix} \cdot
\begin{bmatrix}
1 & 0 & 1 \\
0 & 1 & 1 \\
1 & 0 & 0 \\
0 & 1 & 0
\end{bmatrix}
$$

第 1 行（Q 的第 1 行与 K 的每一列做点积）：
- $(1 \times 1) + (0 \times 0) + (1 \times 1) + (0 \times 0) = 2$
- $(1 \times 0) + (0 \times 1) + (1 \times 0) + (0 \times 1) = 0$
- $(1 \times 1) + (0 \times 1) + (1 \times 0) + (0 \times 0) = 1$

第 2 行（Q 的第 2 行与 K 的每一列做点积）：
- $(0 \times 1) + (1 \times 0) + (0 \times 1) + (1 \times 0) = 0$
- $(0 \times 0) + (1 \times 1) + (0 \times 0) + (1 \times 1) = 2$
- $(0 \times 1) + (1 \times 1) + (0 \times 0) + (1 \times 0) = 1$

$$
QK^T = \begin{bmatrix}
2 & 0 & 1 \\
0 & 2 & 1
\end{bmatrix}
$$

除以 $\sqrt{d_k} = 2$：

$$
S = \frac{QK^T}{\sqrt{d_k}} = \begin{bmatrix}
1.0 & 0.0 & 0.5 \\
0.0 & 1.0 & 0.5
\end{bmatrix}
$$

**步骤 2：Softmax（对每行）**

第 1 行：$[1.0, 0.0, 0.5]$

$$
\begin{aligned}
e^{1.0} &\approx 2.7183 \\
e^{0.0} &= 1.0000 \\
e^{0.5} &\approx 1.6487
\end{aligned}
$$

和：$2.7183 + 1.0000 + 1.6487 \approx 5.3670$

$$
\text{softmax}_1 = \begin{bmatrix}
\frac{2.7183}{5.3670} & \frac{1.0000}{5.3670} & \frac{1.6487}{5.3670}
\end{bmatrix} \approx \begin{bmatrix}
0.5065 & 0.1863 & 0.3072
\end{bmatrix}
$$

第 2 行：$[0.0, 1.0, 0.5]$

$$
\begin{aligned}
e^{0.0} &= 1.0000 \\
e^{1.0} &\approx 2.7183 \\
e^{0.5} &\approx 1.6487
\end{aligned}
$$

和：$1.0000 + 2.7183 + 1.6487 \approx 5.3670$

$$
\text{softmax}_2 = \begin{bmatrix}
\frac{1.0000}{5.3670} & \frac{2.7183}{5.3670} & \frac{1.6487}{5.3670}
\end{bmatrix} \approx \begin{bmatrix}
0.1863 & 0.5065 & 0.3072
\end{bmatrix}
$$

注意力权重矩阵：
$$
A = \text{softmax}(S) = \begin{bmatrix}
0.5065 & 0.1863 & 0.3072 \\
0.1863 & 0.5065 & 0.3072
\end{bmatrix}
$$

**步骤 3：加权求和输出**

$$
\text{Output} = A \times V = \begin{bmatrix}
0.5065 & 0.1863 & 0.3072 \\
0.1863 & 0.5065 & 0.3072
\end{bmatrix} \times
\begin{bmatrix}
1 & 0 & 1 & 0 & 1 \\
0 & 1 & 0 & 1 & 0 \\
1 & 1 & 0 & 0 & 1
\end{bmatrix}
$$

第 1 行输出：
- 列 1: $0.5065 \times 1 + 0.1863 \times 0 + 0.3072 \times 1 = 0.8137$
- 列 2: $0.5065 \times 0 + 0.1863 \times 1 + 0.3072 \times 1 = 0.4935$
- 列 3: $0.5065 \times 1 + 0.1863 \times 0 + 0.3072 \times 0 = 0.5065$
- 列 4: $0.5065 \times 0 + 0.1863 \times 1 + 0.3072 \times 0 = 0.1863$
- 列 5: $0.5065 \times 1 + 0.1863 \times 0 + 0.3072 \times 1 = 0.8137$

第 2 行输出：
- 列 1: $0.1863 \times 1 + 0.5065 \times 0 + 0.3072 \times 1 = 0.4935$
- 列 2: $0.1863 \times 0 + 0.5065 \times 1 + 0.3072 \times 1 = 0.8137$
- 列 3: $0.1863 \times 1 + 0.5065 \times 0 + 0.3072 \times 0 = 0.1863$
- 列 4: $0.1863 \times 0 + 0.5065 \times 1 + 0.3072 \times 0 = 0.5065$
- 列 5: $0.1863 \times 1 + 0.5065 \times 0 + 0.3072 \times 1 = 0.4935$

$$
\text{Output} \approx \begin{bmatrix}
0.8137 & 0.4935 & 0.5065 & 0.1863 & 0.8137 \\
0.4935 & 0.8137 & 0.1863 & 0.5065 & 0.4935
\end{bmatrix}
$$

**验证：输出矩阵形状为 $2 \times 5$（查询数 × 值维度），与预期一致。**

### 6.2 编程题

实现多头注意力（Multi-Head Attention）的前向传播，假设 $\\text{num\\_heads} = 2$，$d_{\\text{model}} = 4$。给定输入 $X$（形状 `(seq_len, batch, d_model)`），分别线性投影到 Q、K、V，每个头的维度 $d_k = d_v = d_{\\text{model}} / \\text{num\\_heads}$。对每个头计算缩放点积注意力，然后将所有头的输出拼接并通过最终线性层。返回输出（形状与输入相同）。

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class MultiHeadAttention(nn.Module):
    """
    多头注意力机制（前向传播）。

    参数：
        d_model: 模型维度
        num_heads: 注意力头数
        dropout: dropout 概率（可选）
    """
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0, f'd_model ({d_model}) 必须能被 num_heads ({num_heads}) 整除'

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_v = d_model // num_heads

        # Q, K, V 的线性投影（合并到一起以减少参数）
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

        # 输出投影
        self.W_o = nn.Linear(d_model, d_model, bias=False)

        self.dropout = nn.Dropout(dropout)

    def forward(self, X):
        """
        参数：
            X: 输入，形状 (seq_len, batch, d_model)

        返回：
            output: 多头注意力输出，形状 (seq_len, batch, d_model)
        """
        seq_len, batch_size, _ = X.shape

        # 1. 线性投影得到 Q, K, V
        Q = self.W_q(X)  # (seq_len, batch, d_model)
        K = self.W_k(X)  # (seq_len, batch, d_model)
        V = self.W_v(X)  # (seq_len, batch, d_model)

        # 2. 拆分为多头
        # 重塑为 (seq_len, batch, num_heads, d_k)
        Q = Q.view(seq_len, batch_size, self.num_heads, self.d_k)
        K = K.view(seq_len, batch_size, self.num_heads, self.d_k)
        V = V.view(seq_len, batch_size, self.num_heads, self.d_v)

        # 转置为 (batch * num_heads, seq_len, d_k) 便于批量计算
        # 或保持：(batch, num_heads, seq_len, d_k)
        Q = Q.permute(1, 2, 0, 3)  # (batch, num_heads, seq_len, d_k)
        K = K.permute(1, 2, 0, 3)
        V = V.permute(1, 2, 0, 3)

        # 3. 缩放点积注意力
        # scores: (batch, num_heads, seq_len, seq_len)
        scale = math.sqrt(self.d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / scale

        # softmax 得到注意力权重
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 加权求和: (batch, num_heads, seq_len, d_v)
        head_outputs = torch.matmul(attn_weights, V)

        # 4. 拼接多头输出
        # (batch, num_heads, seq_len, d_v) → (seq_len, batch, d_model)
        head_outputs = head_outputs.permute(2, 0, 1, 3).contiguous()
        concat = head_outputs.view(seq_len, batch_size, self.d_model)

        # 5. 最终线性投影
        output = self.W_o(concat)

        return output


# ====== 测试 ======
torch.manual_seed(42)

d_model = 4
num_heads = 2
seq_len = 5
batch_size = 3

# 创建输入
X = torch.randn(seq_len, batch_size, d_model)

# 创建多头注意力层
mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)

# 前向传播
output = mha(X)

print(f'输入 X 形状: {X.shape}')
print(f'd_model = {d_model}, num_heads = {num_heads}')
print(f'd_k = d_v = {d_model // num_heads}')
print(f'输出形状: {output.shape}')
print()

# 验证输出形状
assert output.shape == X.shape, \
    f'输出形状 {output.shape} 与输入形状 {X.shape} 不一致！'
print('形状验证通过！')

# 验证参数量
total_params = sum(p.numel() for p in mha.parameters())
print(f'总参数量: {total_params}')

# 手动验证多头拆分的正确性
print(f'\nQ 投影后形状: {mha.W_q(X).shape}')
print(f'拆分多头后 Q 实际形状: (batch={batch_size}, heads={num_heads}, seq={seq_len}, d_k={d_model//num_heads})')

# ====== 额外测试：更大的模型 ======
print('\n===== 更大模型测试 =====')
mha2 = MultiHeadAttention(d_model=16, num_heads=4)
X2 = torch.randn(10, 4, 16)
output2 = mha2(X2)
print(f'输入形状: {X2.shape}')
print(f'输出形状: {output2.shape}')
assert output2.shape == X2.shape
print('形状验证通过！')

输入 X 形状: torch.Size([5, 3, 4])
d_model = 4, num_heads = 2
d_k = d_v = 2
输出形状: torch.Size([5, 3, 4])

形状验证通过！
总参数量: 64

Q 投影后形状: torch.Size([5, 3, 4])
拆分多头后 Q 实际形状: (batch=3, heads=2, seq=5, d_k=2)

===== 更大模型测试 =====
输入形状: torch.Size([10, 4, 16])
输出形状: torch.Size([10, 4, 16])
形状验证通过！
